<div align="center">

# OpenEnv: ESG Compliance & Sustainability

### *RL Training with GRPO + Unsloth*

---

**Train an LLM to act as a corporate ESG strategist using verifiable environment rewards.**

[![GitHub](https://img.shields.io/badge/GitHub-TharunBabu--05%2FOPEN--ENV-blue?logo=github)](https://github.com/TharunBabu-05/OPEN-ENV)
[![License](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

</div>

---

## What You'll Do

<table>
<tr>
<td width="50%">

**Part 1: Setup**
- GPU check & dependency install
- Clone repo & validate imports
- Smoke test the full pipeline

</td>
<td width="50%">

**Part 2: Train & Evaluate**
- Generate training dataset
- Run GRPO training (~45 min on T4)
- Benchmark & visualize results
- Push trained model to HuggingFace

</td>
</tr>
</table>

## Architecture

```
+---------------------------------------------------------+
|  TRAINING CODE (this notebook)                          |
|                                                         |
|  dataset_builder.py  --> prompts (JSONL)                |
|  train_rl.py         --> GRPOTrainer (TRL + Unsloth)    |
|  reward_functions.py --> 4 verifiable reward signals    |
+----------------------------+----------------------------+
                             |
                             v
+----------------------------+----------------------------+
|  ESG ENVIRONMENT (env.py)                               |
|  reset() --> Observation   |  step(action) --> reward   |
|  9 actions | 3 tasks | deterministic grading            |
+---------------------------------------------------------+
```

> **Runtime:** ~50 min total on Colab T4 GPU (free tier)

---
# Part 1: Setup
## Cell 1 -- GPU Check

In [ ]:
# Cell 1 -- GPU Check
import subprocess, sys, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('WARNING: No GPU detected!')
    print('Go to: Runtime > Change runtime type > T4 GPU')
else:
    print(result.stdout)

import torch
assert torch.cuda.is_available(), 'STOP: Enable T4 GPU in Runtime menu first!'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('\nReady to train!')

## Cell 2 -- Install Dependencies

Installs Unsloth (fast 4-bit LoRA), TRL (GRPO trainer), and project dependencies.

In [ ]:
# Cell 2 -- Install dependencies (~3-5 min)
!pip install -q --upgrade huggingface_hub
!pip install -q pydantic openai pyyaml datasets
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q trl peft accelerate bitsandbytes
print('\nAll dependencies installed.')

## Cell 3 -- Clone Repository & Validate

In [ ]:
# Cell 3 -- Clone repo and validate
REPO_URL = 'https://github.com/TharunBabu-05/OPEN-ENV.git'
REPO_DIR = '/content/open_ENV'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repository already present -- pulled latest.')

%cd {REPO_DIR}

from env import ESGEnvironment
from tasks import TASKS, grade_task
from models import Observation, Action

print('\n' + '='*60)
print('  ENVIRONMENT VALIDATED')
print('='*60)
for tid, t in TASKS.items():
    print(f'  {tid:30s} | {t.difficulty:6s} | {t.max_steps} steps | ${t.initial_budget:,.0f}')
print('='*60)

## Cell 4 -- Smoke Test

Validates dataset generation, reward functions, and environment stepping -- **no GPU needed**.

In [ ]:
# Cell 4 -- Smoke test
!python train_rl.py --smoke_test
print('\nIf you see SMOKE TEST PASSED above, proceed to Part 2.')

---
# Part 2: Train & Evaluate
## Cell 5 -- Generate Training Dataset

In [ ]:
# Cell 5 -- Generate dataset
!python dataset_builder.py --episodes 5 --output data/esg_prompts.jsonl

import json
with open('data/esg_prompts.jsonl') as f:
    lines = f.readlines()
print(f'\nDataset: {len(lines)} prompts')
print(f'Tasks:   {set(json.loads(l)["task_id"] for l in lines)}')

## Cell 6 -- GRPO Training (~45 min on T4)

| Config | Value | Why |
|--------|-------|-----|
| Model | Qwen2.5-0.5B-Instruct | Small, fast, fits T4 |
| LoRA rank | 8 | Fewer params = faster |
| Steps | 150 | ~45 min on T4 |
| GRPO rollouts | 4 | Exploration vs speed |

In [ ]:
# Cell 6 -- GRPO training
!python train_rl.py \
    --config train_config_fast.yaml \
    --output_dir outputs/esg_rl_trained

import os
adapter = 'outputs/esg_rl_trained/lora_adapter'
if os.path.exists(adapter):
    print('\n' + '='*60)
    print('  TRAINING COMPLETE')
    print('  Adapter:', os.listdir(adapter))
    print('='*60)
else:
    print('ERROR: Adapter not found.')

## Cell 7 -- Benchmark & Visualize

In [ ]:
# Cell 7 -- Benchmark
import os, json

!python benchmark.py --mode heuristic --seeds 42 43 44 --output results/baseline_heuristic.json

adapter = 'outputs/esg_rl_trained/lora_adapter'
if os.path.exists(adapter):
    !python benchmark.py --mode llm --model_path {adapter} --seeds 42 43 44 --output results/trained.json
    !python plot_results.py --baseline results/baseline_heuristic.json --trained results/trained.json --output_dir results
else:
    !python plot_results.py --baseline results/baseline_heuristic.json --output_dir results

baseline = json.load(open('results/baseline_heuristic.json'))
print(f'\nHeuristic baseline: {baseline["overall_mean_score"]:.3f}')
if os.path.exists('results/trained.json'):
    trained = json.load(open('results/trained.json'))
    print(f'Trained model:      {trained["overall_mean_score"]:.3f}')

from IPython.display import Image, display
for p in ['score_comparison.png', 'reward_history.png', 'esg_metrics.png']:
    if os.path.exists(f'results/{p}'): display(Image(f'results/{p}'))

## Cell 8 -- Push to HuggingFace Hub

> Replace `HF_TOKEN` below with your **write** token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [ ]:
# Cell 8 -- Push to HuggingFace
# Fix Colab's stale huggingface_hub (ANSI import error)
!pip install -q --upgrade huggingface_hub

HF_TOKEN    = 'hf_REPLACE_ME'
HF_USERNAME = 'tharun5054'
MODEL_REPO  = f'{HF_USERNAME}/esg-rl-agent-grpo'

assert HF_TOKEN != 'hf_REPLACE_ME', 'Set your HF_TOKEN above first!'

from huggingface_hub import login, HfApi, create_repo
login(token=HF_TOKEN)
create_repo(MODEL_REPO, exist_ok=True, private=False)

api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path='outputs/esg_rl_trained/lora_adapter',
    repo_id=MODEL_REPO,
    repo_type='model',
)
print(f'Model pushed -> https://huggingface.co/{MODEL_REPO}')

import os
for plot in ['score_comparison.png', 'reward_history.png', 'esg_metrics.png']:
    if os.path.exists(f'results/{plot}'):
        api.upload_file(
            path_or_fileobj=f'results/{plot}',
            path_in_repo=f'results/{plot}',
            repo_id=MODEL_REPO,
            repo_type='model',
        )
print(f'Done! Visit: https://huggingface.co/{MODEL_REPO}')

---
# Summary

| Component | Details |
|-----------|--------|
| **Environment** | ESGEnvironment -- 9 actions, 17-field obs, 3 difficulty levels |
| **Reward** | 4 independent verifiable signals (no LLM-as-judge) |
| **Training** | GRPO via TRL + Unsloth (4-bit LoRA) |
| **Anti-cheat** | NO_ACTION spam penalty, format check, budget abuse detection |
| **Evidence** | Deterministic benchmarks + comparison plots |

**Links:** [GitHub](https://github.com/TharunBabu-05/OPEN-ENV) | [HF Space](https://huggingface.co/spaces/tharun5054/esg-compliance-env) | [OpenEnv](https://github.com/meta-pytorch/OpenEnv)